# SPEC-03: Data Splitting and Leakage Control

This notebook executes the production SPEC-03 implementation. It creates deterministic client-grouped train, validation, and test assignments, verifies exact row/group invariants, writes privacy-safe split artifacts, and demonstrates train-only preprocessing.

The current source is a snapshot. This split measures generalization to unseen clients, not future-period performance.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from machine_learning_project.data.ingestion import load_csv, sha256_file
from machine_learning_project.data.preparation import prepare_supervised_data
from machine_learning_project.data.splitting import split_by_group, write_split_artifacts
from machine_learning_project.data.validation import require_valid_schema
from machine_learning_project.features.preprocessing import build_preprocessor_from_config
from machine_learning_project.utils.config import (
    load_yaml,
    validate_data_config,
    validate_preprocessing_config,
    validate_split_config,
)

## 1. Load and validate contracts

In [2]:
data_config = load_yaml(PROJECT_ROOT / 'configs/data.yaml')['data']
training_config = load_yaml(PROJECT_ROOT / 'configs/training.yaml')['training']
preprocessing_config = load_yaml(PROJECT_ROOT / 'configs/preprocessing.yaml')['preprocessing']
data_config = {**data_config, 'csv_path': str(PROJECT_ROOT / data_config['csv_path'])}

validate_data_config(data_config)
validate_split_config(training_config)
validate_preprocessing_config(preprocessing_config)
print('Data, split, and preprocessing contracts are valid.')

Data, split, and preprocessing contracts are valid.


## 2. Load, fingerprint, and validate source data

The fingerprint is computed from the original bytes and checked again after splitting.

In [3]:
loaded = load_csv(data_config)
require_valid_schema(loaded.dataframe, data_config)
source_hash_before = loaded.sha256
print(f'Rows: {len(loaded.dataframe):,}')
print(f'Clients: {loaded.dataframe[training_config["group_column"]].nunique()}')
print(f'Source SHA-256: {source_hash_before}')

Rows: 30,000
Clients: 32
Source SHA-256: c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396


## 3. Create deterministic grouped partitions

Complete clients are assigned to one partition. Candidate assignments are scored for row-size and class-distribution quality while preserving every configured class.

In [4]:
weights = training_config['split_objective_weights']
split_kwargs = {
    'target_column': data_config['target_column'],
    'group_column': training_config['group_column'],
    'row_key': training_config['row_key'],
    'allowed_labels': data_config['allowed_target_values'],
    'test_size': training_config['test_size'],
    'validation_size': training_config['validation_size'],
    'random_seed': training_config['random_seed'],
    'search_attempts': training_config['search_attempts'],
    'row_ratio_tolerance': training_config['row_ratio_tolerance'],
    'class_ratio_tolerance': training_config['class_ratio_tolerance'],
    'size_weight': weights['size'],
    'class_weight': weights['class'],
    'source_sha256': loaded.sha256,
    'data_schema_version': data_config['schema_version'],
    'split_contract_version': training_config['split_contract_version'],
    'algorithm_version': training_config['split_algorithm_version'],
    'alias_context': training_config['split_alias_context'],
}
splits = split_by_group(loaded.dataframe, **split_kwargs)

partition_summary = pd.DataFrame(splits.manifest['partitions']).T[
    ['row_count', 'actual_row_ratio', 'intended_row_ratio', 'row_ratio_deviation', 'group_count']
]
display(partition_summary)

,row_count,actual_row_ratio,intended_row_ratio,row_ratio_deviation,group_count
train,17670,0.589,0.6,0.011,18
validation,5857,0.195233,0.2,0.004767,7
test,6473,0.215767,0.2,0.015767,7


## 4. Verify exact row, client, and class invariants

In [5]:
frames = {'train': splits.train, 'validation': splits.validation, 'test': splits.test}
row_key = training_config['row_key']
group_key = training_config['group_column']
row_sets = {name: set(frame[row_key]) for name, frame in frames.items()}
group_sets = {name: set(frame[group_key]) for name, frame in frames.items()}

assert row_sets['train'].isdisjoint(row_sets['validation'])
assert row_sets['train'].isdisjoint(row_sets['test'])
assert row_sets['validation'].isdisjoint(row_sets['test'])
assert set().union(*row_sets.values()) == set(loaded.dataframe[row_key])
assert group_sets['train'].isdisjoint(group_sets['validation'])
assert group_sets['train'].isdisjoint(group_sets['test'])
assert group_sets['validation'].isdisjoint(group_sets['test'])
assert set().union(*group_sets.values()) == set(loaded.dataframe[group_key])
assert all(splits.manifest['invariants'].values())

class_counts = pd.DataFrame(
    {name: frame[data_config['target_column']].value_counts() for name, frame in frames.items()}
).fillna(0).astype(int)
assert (class_counts > 0).all().all()
display(class_counts)
print('Passed exact row, client, and class-coverage invariants.')

,train,validation,test
trend_direction,,,
down,9738,3044,3480
stable,3345,1265,1352
up,2557,869,962
new,1380,410,446
flat,650,269,233


Passed exact row, client, and class-coverage invariants.


## 5. Verify order-independent determinism

Shuffling input rows must not change row-to-partition assignments when the stable row key is unchanged.

In [6]:
shuffled_source = loaded.dataframe.sample(frac=1, random_state=999)
shuffled_splits = split_by_group(shuffled_source, **split_kwargs)
assert splits.assignments.equals(shuffled_splits.assignments)
assert splits.manifest == shuffled_splits.manifest
print('Passed deterministic assignment check after input reordering.')

Passed deterministic assignment check after input reordering.


## 6. Write privacy-safe artifacts

The manifest contains masked client aliases. The assignment table contains one-way row-key hashes rather than raw `content_id` values.

In [7]:
manifest_path, assignments_path = write_split_artifacts(
    splits,
    PROJECT_ROOT / training_config['split_manifest_path'],
    PROJECT_ROOT / training_config['split_assignments_path'],
)
display(pd.DataFrame(splits.assignments).head())
display(json.loads(manifest_path.read_text(encoding='utf-8'))['invariants'])

assignment_artifact = pd.read_csv(assignments_path, dtype=str)
manifest_artifact = json.loads(manifest_path.read_text(encoding='utf-8'))
raw_clients = set(loaded.dataframe[group_key].astype(str))
raw_rows = set(loaded.dataframe[row_key].astype(str))
published_clients = set(assignment_artifact['group_alias'])
published_rows = set(assignment_artifact['row_key_hash'])
assert raw_clients.isdisjoint(published_clients)
assert raw_rows.isdisjoint(published_rows)
assert all('groups' not in details for details in manifest_artifact['partitions'].values())
print('Passed split-artifact privacy check.')

,row_key_hash,group_alias,partition,split_contract_version
0,000773aeb4a354f73be6504899504cbca4d3f8de92ff8a...,ef46afc2db406154,validation,1.0
1,0008cf4bc56cafd69dc8d59fbddc24ebf6220da3bc4c68...,1568b3d35c3585ad,test,1.0
2,0009ff0c3c2c51c727933cac10078a612a675e8d929ebf...,219b56b648b107cc,train,1.0
3,000b4d226a398e15c172fef5e73fefa7595623c4e29236...,bc389c2007f09dc0,test,1.0
4,000d4479c3915fd5fd39c242f4622079ade15748519f50...,ef46afc2db406154,validation,1.0


{'class_coverage': True,
 'group_complete': True,
 'group_disjoint': True,
 'row_complete': True,
 'row_disjoint': True}

Passed split-artifact privacy check.


## 7. Demonstrate split-before-fit preprocessing

The transformer is fitted once on training features. Validation and test data only call `transform`, preventing their medians, scales, or categories from influencing learned state.

In [8]:
prepared = {
    name: prepare_supervised_data(
        frame,
        data_config,
        feature_contract_version=preprocessing_config['feature_contract_version'],
    )
    for name, frame in frames.items()
}
transformer = build_preprocessor_from_config(
    data_config['numeric_columns'],
    data_config['categorical_columns'],
    preprocessing_config,
)
matrices = {
    'train': transformer.fit_transform(prepared['train'].features),
    'validation': transformer.transform(prepared['validation'].features),
    'test': transformer.transform(prepared['test'].features),
}
assert len({matrix.shape[1] for matrix in matrices.values()}) == 1
display(pd.DataFrame({name: matrix.shape for name, matrix in matrices.items()}, index=['rows', 'features']).T)

,rows,features
train,17670,46
validation,5857,46
test,6473,46


## 8. Final source-integrity check and remaining work

In [9]:
assert sha256_file(loaded.source_path) == source_hash_before
print('Raw CSV fingerprint is unchanged.')
print('Remaining SPEC-03 work: separate development selection from final-test evaluation')
print('and add explicit first/repeated test-access audit policy.')

Raw CSV fingerprint is unchanged.
Remaining SPEC-03 work: separate development selection from final-test evaluation
and add explicit first/repeated test-access audit policy.
